# Module 2 — Delta Lake (ACID, MERGE, Time Travel, OPTIMIZE/VACUUM)
Exam domain: **Databricks Tooling**

Runs standalone in Google Colab — no Databricks account needed.

In [ ]:
# 1. Install PySpark + Delta Lake
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
# 2. Create Spark session with Delta support
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql import functions as F

builder = (SparkSession.builder
    .appName("Module2-DeltaLake")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

## ACID transactions
Every write to a Delta table is atomic: it either fully commits or fully fails.
Concurrent readers never see a partial write.

In [ ]:
customers = [(1, "Alice", "gold"), (2, "Bob", "silver"), (3, "Cara", "silver")]
df = spark.createDataFrame(customers, ["id", "name", "tier"])
df.write.format("delta").mode("overwrite").save("/content/lake/customers")
spark.read.format("delta").load("/content/lake/customers").show()

## MERGE INTO (upsert)
Classic pattern for landing CDC-style changes: update existing rows, insert new ones.

In [ ]:
updates = [(2, "Bob", "gold"), (4, "Dana", "silver")]  # id=2 upgraded, id=4 is new
updates_df = spark.createDataFrame(updates, ["id", "name", "tier"])

target = DeltaTable.forPath(spark, "/content/lake/customers")

(target.alias("t")
    .merge(updates_df.alias("s"), "t.id = s.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.read.format("delta").load("/content/lake/customers").orderBy("id").show()

**Ambiguous MERGE:** if the join condition matches more than one source row per
target row, Delta raises an error at execution time — try it with a duplicated `id`
in `updates` to see the real exception.

## Time travel
Every write creates a new table version. You can query any historical version by
number or timestamp.

In [ ]:
history_df = target.history()
history_df.select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
# Read version 0 (before the MERGE) and compare with current
v0 = spark.read.format("delta").option("versionAsOf", 0).load("/content/lake/customers")
current = spark.read.format("delta").load("/content/lake/customers")
print("Version 0:")
v0.orderBy("id").show()
print("Current:")
current.orderBy("id").show()

## OPTIMIZE and ZORDER
Compacts small files into larger ones and (optionally) co-locates data by column
to speed up filters. OSS delta-spark supports these via the Python API.

In [ ]:
target.optimize().executeCompaction()
target.optimize().executeZOrderBy("tier")
print("Optimize + Z-Order complete")

## VACUUM
Removes files no longer referenced by the table's active versions, after the
retention window. Retention 0 is dangerous: it deletes files that time-travel
queries and concurrent readers might still need.

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
target.vacuum(0)  # retention 0 hours — for the lab only, never in production
print("Vacuumed. Version 0 is now unreadable — try re-running the versionAsOf(0) read above.")

## Notes
- `mergeSchema` is intentionally **not** used here — it's covered in Module 6 (schema evolution).
- In production, VACUUM should keep the default 7-day retention so time travel and
  concurrent long-running reads stay safe.